## 🏗️ LeetCode 435: Non-overlapping Intervals
---

<div style="padding: 15px; border-left: 8px solid #4CAF50; background-color: #e8f5e9; color: #1b5e20; border-radius: 4px;">
    <strong>The Core Insight:</strong> To minimize removals = maximize intervals we can keep. Greedy rule: when two intervals overlap, always keep the one that ends earlier — it leaves the most room for future intervals. Sort by END time and apply this greedy choice.
</div>

### 🛠️ The Mathematical Model

Minimum removals = n - maximum non-overlapping intervals we can keep.
Greedy: always select the interval with the earliest end time from the remaining non-overlapping candidates.

$$\text{min removals} = n - \text{count of non-overlapping intervals (greedy)}$$

---

### 📋 Problem

Find the minimum number of intervals to remove to make the remaining intervals non-overlapping.

**Example 1:**
```
Input:  [[1,2],[2,3],[3,4],[1,3]]
Output: 1   (remove [1,3])
```

**Example 2:**
```
Input:  [[1,2],[1,2],[1,2]]
Output: 2   (keep one, remove two)
```

**Constraints:** 1 ≤ intervals.length ≤ 10⁵ | -5×10⁴ ≤ start_i < end_i ≤ 5×10⁴

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Activity Selection</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"I can only do one activity at a time. To fit the MOST activities in a day, always pick the activity that finishes earliest — it frees up time for the next one soonest."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Sort by end time. Keep intervals that start ≥ last_end. Count removals = n - kept.</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Conference Room Optimizer</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"One conference room, many meetings. Cancel the minimum meetings so the rest don't conflict. Always cancel the meeting that ends later when two conflict — the shorter meeting leaves more room."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">When two intervals overlap, the one ending later is removed (or equivalently: the one ending earlier is kept).</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> Trying all subsets of intervals is O(2^n). The greedy approach runs in O(n log n) — sort once, scan once.
</div>

## 🐢 Approach 1: Brute Force — $O(2^n)$

In [ ]:
from itertools import combinations

def eraseOverlapIntervals_brute(intervals):
    """
    Brute Force: Try all subsets, find largest non-overlapping subset
    Time: O(2^n * n) | Space: O(n)
    """
    def is_non_overlapping(subset):
        subset = sorted(subset)
        for i in range(1, len(subset)):
            if subset[i][0] < subset[i-1][1]:   # overlap check
                return False
        return True

    n = len(intervals)
    max_kept = 0
    for size in range(n, 0, -1):
        for subset in combinations(intervals, size):
            if is_non_overlapping(list(subset)):
                max_kept = size
                break
        if max_kept:
            break
    return n - max_kept


# Small test only (exponential — don't run on large input)
print(eraseOverlapIntervals_brute([[1,2],[2,3],[3,4],[1,3]]))   # Expected: 1

## 🔬 Complexity Analysis: $O(2^n)$ vs. $O(n \log n)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> Greedy correctness: when two intervals overlap, keeping the one that ends earlier is always optimal — it cannot reduce the number of future intervals we can keep (it ends sooner, freeing up more timeline). This greedy choice is provably optimal for the Activity Selection Problem.
</div>

---

## 📉 Why Brute Force Fails: The $O(2^n)$ Trap

2^n subsets for n = 50 intervals: ~10¹⁵ subsets. Infeasible.

| Input Type | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **n = 20** | ~1 million subsets | Exponential growth |
| **n = 50** | ~10¹⁵ subsets | Completely infeasible |

---

## 🚀 The Optimal Approach: $O(n \log n)$

Sort by END time. Greedily keep intervals. For each interval:
- If `start >= last_end`: no overlap — keep it, update `last_end = end`
- If `start < last_end`: overlap — remove it (increment counter)

### The Key Lifecycle Rule
1. **Sort by END time** (not start time — this is the key difference from Merge Intervals)
2. **Greedily keep** intervals that start after the last kept interval's end
3. **Count removals** = number of skipped intervals

---

## ✅ Mathematical Proof (Greedy Correctness)

Suppose optimal solution keeps interval A (ends late) over interval B (ends early, overlaps A). Swap A for B — B ends earlier, so it cannot conflict with more future intervals than A does. Therefore swapping never makes the solution worse. The greedy choice (keep earliest-ending) is optimal. ∎

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Sort by END (not start). Keep intervals that don't overlap the last kept. Count removals. The greedy proof shows that keeping the earliest-ending interval is always optimal.
</div>

## 🚀 Approach 2: Greedy — Sort by End Time — $O(n \log n)$
---

Instead of trying all subsets, we use a **greedy rule: always keep the interval that ends earliest**.

As we iterate:
1. Sort by END time (critical — not start time)
2. Track `last_end = -∞`
3. For each interval: if `start >= last_end`, keep it (update last_end). Else, remove it (increment counter).

In [ ]:
def eraseOverlapIntervals(intervals: list[list[int]]) -> int:
    """
    Optimal: Greedy — sort by end time, keep earliest-ending
    Time: O(n log n) | Space: O(1)
    """
    intervals.sort(key=lambda x: x[1])   # sort by END time (not start!)
    removals = 0
    last_end = float('-inf')

    for start, end in intervals:
        if start >= last_end:    # no overlap — keep this interval
            last_end = end
        else:
            removals += 1        # overlap — remove (we implicitly keep the earlier-ending one)

    return removals


print("Optimal:", eraseOverlapIntervals([[1,2],[2,3],[3,4],[1,3]]))   # Expected: 1
print("Optimal:", eraseOverlapIntervals([[1,2],[1,2],[1,2]]))          # Expected: 2
print("Optimal:", eraseOverlapIntervals([[1,2],[2,3]]))                # Expected: 0

## 🎤 5 Interview Q&A

### Q1: Why sort by end time instead of start time?

**Answer:** Greedy correctness — we want to maximize the number of intervals we can keep. The interval that ends earliest leaves the most remaining timeline for future intervals. Sorting by end time ensures that when we scan left to right, we encounter the earliest-ending interval first and greedily keep it. Sorting by start time does NOT have this property.

---

### Q2: Is this the Activity Selection Problem?

**Answer:** Yes — exactly. The Activity Selection Problem asks: given activities with start and end times, select the maximum number of non-conflicting activities. The greedy solution is identical: sort by end time, greedily select each activity that starts after the previous one ends. LC #435 is the complement: min removals = n - max kept activities.

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** O(n log n) — dominated by the sort. The greedy scan is O(n) — each interval is visited exactly once, and `last_end` and `removals` are updated in O(1). Overall: O(n log n + n) = O(n log n).

---

### Q4: How does this differ from Merge Intervals (LC #56) sorting?

**Answer:** LC #56 sorts by START time — we need adjacent overlapping intervals to be adjacent in sorted order for the merge to work. LC #435 sorts by END time — we need to greedily select intervals that end earliest. Two different interval problems, two different sort keys. Start = merge. End = greedy removal/selection.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) No overlapping — removals = 0. Last_end updates each iteration, start ≥ last_end always true. (2) All same interval — all overlap except first; removals = n-1. (3) Touching intervals (end = next start) — `start >= last_end` with `>=` correctly treats touching as non-overlapping (problem uses open/closed convention — verify). (4) Single interval — for loop runs once, no removals.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Activity Selection Problem** | Classic CS problem: maximum non-conflicting activities from n activities with start/end times. Greedy solution: sort by end, select greedily. |
| **Greedy Algorithm** | Makes the locally optimal choice at each step with the goal of finding a global optimum |
| **Greedy Correctness** | Proof that the greedy choice (keep earliest-ending) is always optimal — shown by exchange argument |
| **Sort by End** | The non-obvious sort key for greedy interval selection problems — enables the Activity Selection greedy |
| **Exchange Argument** | Proof technique: show that swapping the greedy choice for any other never improves the solution |
| **Min Removals** | n minus the maximum number of non-overlapping intervals we can keep |
| **last_end** | Tracks the end time of the last kept interval — the current "right boundary" of the selected set |
| **Touching Intervals** | Intervals that share only an endpoint (e.g., [1,2] and [2,3]) — typically considered non-overlapping |

## 💼 The Citi Narrative

**Context:** Citi's infrastructure team had a maintenance calendar with 80 approved maintenance windows for a given weekend — but the network could only support 40 simultaneous maintenance operations (capacity constraint). Many windows overlapped. The goal: cancel the minimum number of windows.

**Scenario:** 80 maintenance windows, sorted by end time. Greedy: keep the window that ends earliest when conflicts arise. In practice, this maximized the number of maintenance operations completed over the weekend — cancelled windows could be rescheduled next weekend.

**How this pattern applied:** The greedy algorithm processed 80 windows in O(80 log 80) ≈ 500 comparisons. For each conflict, the window ending later was cancelled (it blocked more future windows). The result: minimum cancellations with maximum maintenance throughput.

**Impact:** Weekend maintenance scheduling went from manual negotiation (which windows to cancel) to automated greedy optimization. The algorithm consistently found 5-10% more maintenance slots per weekend compared to manual approaches, accelerating the patching backlog reduction.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Given intervals, find the MAXIMUM number of
# non-overlapping intervals you can keep (complement of LC #435).
# -------------------------------------------------------

def maxNonOverlapping(intervals):
    # Your solution here — greedy, sort by end
    pass


# Test
print(maxNonOverlapping([[1,2],[2,3],[3,4],[1,3]]))   # Expected: 3 (keep [1,2],[2,3],[3,4])
print(maxNonOverlapping([[1,2],[1,2],[1,2]]))          # Expected: 1

## 🎯 Summary: Key Takeaways

### The Pattern
**Greedy — Sort by End** — Always keep the interval that ends earliest; minimizes future conflicts

### When to Use It
- ✅ Minimize removals to make intervals non-overlapping
- ✅ Maximize number of non-conflicting activities (Activity Selection)
- ✅ Scheduling problems where "fewest conflicts" is the goal
- ❌ **Don't use when:** You need to merge overlapping intervals — use LC #56 (sort by start)

### Complexity
| Approach | Time | Space |
|----------|------|-------|
| Brute Force (all subsets) | $O(2^n)$ | $O(n)$ |
| Optimal (Greedy, sort by end) | $O(n \log n)$ | $O(1)$ |

### Interview Confidence Checklist
- [ ] Can explain why sort by END (not start) for this problem
- [ ] Can state the greedy correctness proof (exchange argument)
- [ ] Can write the solution in 3 minutes from memory
- [ ] Can relate to Activity Selection Problem by name
- [ ] Can contrast with LC #56 sorting approach

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Non-overlapping Intervals and you've mastered the Activity Selection greedy — one of the most elegant algorithm design insights. 🚀